# Network Attack Detection - Data Preprocessing & Cleaning
Ce notebook détaille le pipeline complet d'ingestion, d'assainissement et de normalisation du dataset **CIC-IDS-2017**.
L'objectif est de transformer les captures réseau brutes en données propres et prêtes pour la classification supervisée.

In [7]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Configuration de l'affichage Pandas
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


## 1. Chargement et fusion des captures réseau
Le dataset CIC-IDS-2017 est partitionné en 8 fichiers CSV (du lundi au vendredi).
Nous les chargeons et les combinons en un seul grand DataFrame. 
Une vérification de la casse des noms de fichiers est intégrée pour éviter les erreurs d'exécution sous Windows/Linux.
Un échantillonnage optionnel (`sample_fraction`) est prévu pour tester les calculs rapidement sur machines à mémoire modeste.

In [9]:
def load_and_merge_datasets(dataset_dir='../dataset/', sample_fraction=None):
    """
    Charge les 8 fichiers CSV du CIC-IDS-2017 et les fusionne en un DataFrame unique.
    Gère l'encodage cp1252/latin1 pour éviter les erreurs de décodage des caractères web.
    """
    files = [
        'Monday-WorkingHours.pcap_ISCX.csv',
        'Tuesday-WorkingHours.pcap_ISCX.csv',
        'Wednesday-workingHours.pcap_ISCX.csv',
        'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
        'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
        'Friday-WorkingHours-Morning.pcap_ISCX.csv',
        'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
        'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
    ]
    
    dfs = []
    available_files = {f.lower(): f for f in os.listdir(dataset_dir) if f.endswith('.csv')} if os.path.exists(dataset_dir) else {}
    
    for file in files:
        actual_name = available_files.get(file.lower())
        if actual_name:
            file_path = os.path.join(dataset_dir, actual_name)
            print(f"Chargement : {actual_name}")
            # encodage cp1252 avec repli sur latin1 pour charger sans bloquer sur les caractères corrompus
            try:
                df_temp = pd.read_csv(file_path, encoding='cp1252', low_memory=False)
            except UnicodeDecodeError:
                df_temp = pd.read_csv(file_path, encoding='latin1', low_memory=False)
            dfs.append(df_temp)
        else:
            print(f"Attention : Fichier non trouvé : {file}")
            
    if not dfs:
        raise FileNotFoundError(f"Aucun fichier CSV valide trouvé dans {dataset_dir}")
        
    df = pd.concat(dfs, ignore_index=True)
    
    # Nettoyage des espaces superflus dans les noms de colonnes
    df.columns = df.columns.str.strip()
    
    if sample_fraction is not None and 0 < sample_fraction < 1.0:
        df = df.sample(frac=sample_fraction, random_state=42).reset_index(drop=True)
        print(f"Échantillonnage appliqué : {len(df)} lignes retenues.")
        
    return df

# Exécution du chargement
df_raw = load_and_merge_datasets(dataset_dir='../dataset/', sample_fraction=None)
print("Dimensions brutes :", df_raw.shape)

Chargement : Monday-WorkingHours.pcap_ISCX.csv
Chargement : Tuesday-WorkingHours.pcap_ISCX.csv
Chargement : Wednesday-workingHours.pcap_ISCX.csv
Chargement : Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Chargement : Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Chargement : Friday-WorkingHours-Morning.pcap_ISCX.csv
Chargement : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Chargement : Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Dimensions brutes : (3119345, 85)


## 2. Nettoyage des données (Data Cleaning)
Le trafic réseau engendre des anomalies numériques :
1. **Valeurs infinies (`+inf`, `-inf`)** : issues de divisions par zéro lors du calcul des taux de paquets/octets par seconde. Elles sont converties en `NaN`.
2. **Colonnes constantes ou quasi-nulles** : les attributs ayant $\ge 99\%$ de valeurs nulles (ex. drapeaux TCP rares : *RST Flag*, *CWE Flag*) n'apportent aucun pouvoir discriminant et sont éliminés.
3. **Lignes incomplètes** : suppression des lignes contenant encore des `NaN` via `dropna()`.

In [10]:
def clean_data(df, zero_threshold=99.0):
    """
    Assainit le jeu de données en traitant les infinis, colonnes invariantes et NaN.
    """
    print(f"Nombre de colonnes initial : {df.shape[1]}")
    
    # 1. Remplacement des infinis par NaN
    df_clean = df.replace([np.inf, -np.inf], np.nan)
    
    # 2. Suppression des colonnes avec >= 99% de zéros
    total_rows = df_clean.shape[0]
    cols_to_drop = [
        col for col in df_clean.columns 
        if (df_clean[col] == 0).sum() / total_rows * 100 >= zero_threshold
    ]
    
    print(f"Colonnes supprimées (≥ {zero_threshold}% de zéros) : {len(cols_to_drop)}")
    for col in cols_to_drop:
        print(f" - {col}")
        
    df_clean = df_clean.drop(columns=cols_to_drop)
    
    # 3. Suppression des valeurs manquantes
    rows_before = df_clean.shape[0]
    df_clean = df_clean.dropna().copy()
    rows_after = df_clean.shape[0]
    
    print(f"Lignes supprimées contenant des NaN : {rows_before - rows_after} ({(rows_before - rows_after)/rows_before*100:.2f} %)")
    print(f"Dimensions finales nettoyées : {df_clean.shape}")
    
    return df_clean

df_cleaned = clean_data(df_raw)

Nombre de colonnes initial : 85
Colonnes supprimées (≥ 99.0% de zéros) : 0
Lignes supprimées contenant des NaN : 291469 (9.34 %)
Dimensions finales nettoyées : (2827876, 85)


## 3. Encodage des étiquettes (Label Encoding)
La colonne `Label` catégorise 15 types d'activités (trafic sain `BENIGN` ou 14 attaques spécifiques).
* Les étiquettes textuelles sont normalisées pour traiter les corruptions de caractères (notamment sur les attaques Web).
* Pour répondre à l'objectif de détection binaire d'intrusion, la cible est projetée :
  * **1** : Trafic sain (`BENIGN`).
  * **0** : Trafic malveillant (Attaque).

In [11]:
import re

def encode_labels(df, binary=True):
    """
    Normalise les libellés et convertit la variable cible en format binaire ou multi-classes.
    """
    df_copy = df.copy()
    
    # 1. Supprime les espaces invisibles avant et après les noms de colonnes
    df_copy.columns = df_copy.columns.str.strip()
    
    # 2. Détecte la colonne du label (sensible ou insensible à la casse)
    label_col = None
    for col in df_copy.columns:
        if col.strip().lower() == 'label':
            label_col = col
            break
            
    if label_col is None:
        raise KeyError(
            f"Colonne cible introuvable. Colonnes disponibles : {list(df_copy.columns[-5:])}"
        )

    def norm(s):
        return ' '.join(re.sub(r'[^A-Za-z\- ]', ' ', str(s)).split())

    label_names = [
        'BENIGN', 'FTP-Patator', 'SSH-Patator', 'DoS slowloris', 'DoS Slowhttptest',
        'DoS Hulk', 'DoS GoldenEye', 'Heartbleed', 'Web Attack Brute Force',
        'Web Attack XSS', 'Web Attack Sql Injection', 'Infiltration', 'Bot', 'DDoS', 'PortScan'
    ]
    label_map = {norm(name): idx for idx, name in enumerate(label_names)}
    
    encoded_label = df_copy[label_col].map(lambda s: label_map.get(norm(s)))
    
    # Vérification d'absence de libellés inconnus
    unknown_count = encoded_label.isna().sum()
    assert unknown_count == 0, f"{unknown_count} lignes possèdent un label non répertorié !"
    
    df_copy['Label'] = encoded_label.astype(int)
    
    # Si la colonne d'origine avait un autre nom que 'Label', on la retire
    if label_col != 'Label':
        df_copy = df_copy.drop(columns=[label_col])
        
    X = df_copy.drop(columns=['Label'])
    y = df_copy['Label']
    
    if binary:
        # 1 = BENIGN (classe 0), 0 = Attaque
        y = (y == 0).astype(int)
        
    return X, y

# Exécution
X, y = encode_labels(df_cleaned, binary=True)

# Affichage de la distribution
print("Distribution de la cible binaire (1 = BENIGN, 0 = Attaque) :")
print(y.value_counts(normalize=True) * 100)

Distribution de la cible binaire (1 = BENIGN, 0 = Attaque) :
Label
1    80.318939
0    19.681061
Name: proportion, dtype: float64


## 4. Découpage Train/Test et Mise à l'échelle (Feature Scaling)
* **Stratification** : Le dataset étant déséquilibré, `stratify=y` préserve la proportion exacte d'attaques entre le jeu d'entraînement (70%) et de test (30%).
* **Prévention du Data Leakage** : Le `StandardScaler` apprend ses paramètres (moyenne et variance) **uniquement** sur les données d'entraînement (`fit_transform`) et applique ensuite cette transformation sur les données de test (`transform`).

In [12]:
def split_and_scale(X, y, test_size=0.3, random_state=1):
    """
    Découpe le dataset et applique une standardisation stricte sans fuite de données.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    scaler = StandardScaler()
    num_cols = X_train.select_dtypes(include='number').columns
    
    # Fit uniquement sur le train
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train[num_cols]), columns=num_cols)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols)
    
    print(f"Jeu d'entraînement : {X_train_scaled.shape}")
    print(f"Jeu de test         : {X_test_scaled.shape}")
    
    return X_train_scaled, X_test_scaled, y_train.values, y_test.values

X_train, X_test, y_train, y_test = split_and_scale(X, y, test_size=0.3, random_state=1)

Jeu d'entraînement : (1979513, 80)
Jeu de test         : (848363, 80)
